# 🌾 RICE VISION AI — TRỰC QUAN HÓA & PHÂN TÍCH CHUYÊN SÂU KẾT QUẢ BENCHMARK

Sổ tay này được thiết kế chuyên biệt để **trực quan hóa toàn diện, sinh động và khoa học** kết quả kiểm nghiệm của hệ thống Rice Vision AI (bao gồm thuật toán **Bayesian Few-Shot Calibration - FromThanh SOTA** và các phương pháp đối chứng).

### 🎯 Nội dung trực quan hóa:
1. **Bảng so sánh đối đầu Đa phương pháp**: Đối chiếu $R^2$, MAE, RMSE, MAPE, Dung sai $\le 3\%, \le 5\%, \le 10\%$ giữa **Bayesian Hybrid**, **Extra Trees**, **Thuần Hình Học (Geometry)** và **Cân Mẫu (Weight)**.
2. **Biểu đồ Tương quan & Độ chính xác (Accuracy Analytics)**: Scatter Plot ($y$ vs $\hat{y}$), Đồ thị Thỏa hiệp Bland-Altman, Đường cong tích lũy sai số (Cumulative Error Curves).
3. **Biểu đồ Phân phối Phần dư & Độ bất định (Uncertainty Analytics)**: Phân phối phần dư sai số (Residual KDE), Bề rộng và Tỷ lệ bao phủ thực nghiệm của Khoảng tin cậy 90% (CI90 Coverage Probability), Phân tích đa nhánh con (Ảnh thuần vs Có cân).
4. **Phân tích theo Kích thước Cốc (Lot Size Breakdown)**: Đánh giá độ thích nghi sai số theo từng nhóm đường kính ly ($D = 17.8\text{ mm}, 22.5\text{ mm}, 32.6\text{ mm}$) và theo chiều cao lớp lúa.
5. **Thư viện Ảnh Thực tế (Visual Sample Gallery)**: Hiển thị ảnh chụp trực quan của Top mẫu chuẩn xác nhất và Top mẫu sai số (Outliers), kèm công cụ soi chi tiết từng mẫu bất kỳ.
6. **Tự động xuất bộ sưu tập hình ảnh chất lượng cao (300 DPI)** và đóng gói tệp ZIP tải về một chạm trên Google Colab.


In [ ]:
# =============================================================================
# 1. KHỞI TẠO MÔI TRƯỜNG & CẤU HÌNH TRỰC QUAN HÓA (COLAB / LOCAL HYBRID)
# =============================================================================
import os
import sys
import time
import json
import subprocess
from pathlib import Path

# 1.1 Tự động phát hiện môi trường Google Colab
IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    print('⚡ Phát hiện môi trường Google Colab!')
    if not Path('/content/drive').exists():
        from google.colab import drive
        drive.mount('/content/drive')

    # Cài đặt thư viện bổ trợ cần thiết
    try:
        import openpyxl
        import seaborn
    except ImportError:
        print('📦 Đang cài đặt thư viện đồ họa & excel (openpyxl, seaborn)...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'openpyxl', 'seaborn', 'matplotlib', 'pillow'])

    # Định vị thư mục test_data trên Google Drive hoặc thư mục làm việc
    candidates = [
        Path('/content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES/AI_SERVICES/test_data'),
        Path('/content/drive/MyDrive/MAIN_SOURCES/AI_SERVICES/test_data'),
        Path('/content/drive/MyDrive/AI_SERVICES/test_data'),
        Path('/content/AI_SERVICES/test_data'),
        Path.cwd() if Path.cwd().name == 'test_data' else Path.cwd() / 'test_data',
        Path.cwd()
    ]
    TEST_DATA_DIR = None
    for c in candidates:
        if (c / '1_Raw_Images').is_dir() and ((c / 'benchmark_results').is_dir() or (c / '2_Manual_Records').is_dir()):
            TEST_DATA_DIR = c
            break
    if TEST_DATA_DIR is None:
        matches = list(Path('/content/drive/MyDrive').glob('**/AI_SERVICES/test_data'))
        if matches:
            TEST_DATA_DIR = matches[0]
    if TEST_DATA_DIR is None:
        TEST_DATA_DIR = Path('/content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES/AI_SERVICES/test_data')
else:
    print('💻 Phát hiện môi trường Local Machine.')
    CURRENT_DIR = Path.cwd()
    if CURRENT_DIR.name == 'test_data':
        TEST_DATA_DIR = CURRENT_DIR
    elif (CURRENT_DIR / 'test_data').is_dir():
        TEST_DATA_DIR = CURRENT_DIR / 'test_data'
    elif (CURRENT_DIR / 'AI_SERVICES' / 'test_data').is_dir():
        TEST_DATA_DIR = CURRENT_DIR / 'AI_SERVICES' / 'test_data'
    else:
        matches = [p for p in CURRENT_DIR.parents if (p / 'AI_SERVICES' / 'test_data').is_dir()]
        TEST_DATA_DIR = matches[0] / 'AI_SERVICES' / 'test_data' if matches else CURRENT_DIR

RAW_IMG_DIR = TEST_DATA_DIR / '1_Raw_Images'
MANUAL_EXCEL = TEST_DATA_DIR / '2_Manual_Records' / 'manual_data.xlsx'
BENCHMARK_DIR = TEST_DATA_DIR / 'benchmark_results'
OUTPUT_PLOTS_DIR = BENCHMARK_DIR / 'visual_gallery'
OUTPUT_PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# 1.2 Thiết lập giao diện đồ họa chuẩn báo cáo khoa học
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from PIL import Image

matplotlib.rcParams['figure.dpi'] = 150
matplotlib.rcParams['savefig.dpi'] = 300
matplotlib.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'Segoe UI', 'Helvetica']
matplotlib.rcParams['axes.edgecolor'] = '#cbd5e1'
matplotlib.rcParams['axes.linewidth'] = 0.8
sns.set_theme(style='whitegrid', font_scale=1.05)

print(f'✅ Đã liên kết thư mục dữ liệu kiểm thử: {TEST_DATA_DIR}')
print(f'📁 Thư mục xuất ảnh đồ thị: {OUTPUT_PLOTS_DIR}')


In [ ]:
# =============================================================================
# 2. NẠP DỮ LIỆU KẾT QUẢ BENCHMARK & GHÉP NỐI THÔNG SỐ VẬT LÝ
# =============================================================================
# Tìm file kết quả benchmark mới nhất
res_candidates = [
    BENCHMARK_DIR / 'benchmark_100_results.xlsx',
    TEST_DATA_DIR / 'benchmark_100_results.xlsx',
    BENCHMARK_DIR / 'benchmark_100_results.csv',
    TEST_DATA_DIR / 'benchmark_100_results.csv'
]
res_file = None
for cand in res_candidates:
    if cand.is_file():
        res_file = cand
        break

if res_file is None:
    # Tìm bất kỳ file kết quả nào trong benchmark_results
    xlsx_files = list(BENCHMARK_DIR.glob('benchmark_100_results*.xlsx'))
    if xlsx_files:
        res_file = sorted(xlsx_files, key=lambda f: f.stat().st_mtime, reverse=True)[0]

if res_file is None:
    raise FileNotFoundError(f'Không tìm thấy file kết quả benchmark trong {BENCHMARK_DIR}.')

print(f'📊 Đang nạp kết quả từ: {res_file.name}')
if res_file.suffix.lower() == '.csv':
    df_bench = pd.read_csv(res_file)
else:
    df_bench = pd.read_excel(res_file)

# Đọc file hồ sơ đo tay (Manual Records) nếu có để ghép thêm thông số vật lý
if MANUAL_EXCEL.is_file():
    df_man = pd.read_excel(MANUAL_EXCEL)
    df_bench['Sample_ID'] = df_bench['Sample_ID'].astype(str).str.strip()
    df_man['Sample_ID'] = df_man['Sample_ID'].astype(str).str.strip()
    
    # Ghép thông số bổ sung: Kích thước ly, mức hụt, thể tích khối
    cols_to_add = [c for c in df_man.columns if c not in df_bench.columns or c == 'Sample_ID']
    df_merged = pd.merge(df_bench, df_man[cols_to_add], on='Sample_ID', how='left')
else:
    df_merged = df_bench.copy()

# Tính toán các đại lượng vật lý chuẩn hóa
if 'Inner_Diameter_mm' in df_merged.columns and 'Container_Height_mm' in df_merged.columns:
    h_cont = df_merged['Container_Height_mm'].fillna(0.0)
    h_empty = df_merged['Empty_Height_mm'].fillna(0.0) if 'Empty_Height_mm' in df_merged.columns else 0.0
    df_merged['Rice_Height_mm'] = np.maximum(0.0, h_cont - h_empty)
    radius_mm = df_merged['Inner_Diameter_mm'] / 2.0
    df_merged['Bulk_Volume_mm3'] = np.pi * (radius_mm ** 2) * df_merged['Rice_Height_mm']
    
    # Phân nhóm kích cỡ ly (Lot Group)
    def assign_lot_group(diam):
        if pd.isna(diam): return 'Unknown'
        d = float(diam)
        if d < 20.0: return f'Ly nhỏ ({d:.1f}mm)'
        elif d < 28.0: return f'Ly vừa ({d:.1f}mm)'
        else: return f'Ly lớn ({d:.1f}mm)'
    df_merged['Lot_Group'] = df_merged['Inner_Diameter_mm'].apply(assign_lot_group)

# Tính lại sai số chuẩn hóa nếu thiếu
if 'Error' not in df_merged.columns and 'Final_Estimate' in df_merged.columns:
    df_merged['Error'] = df_merged['Final_Estimate'] - df_merged['Actual_Count']
if 'Abs_Error' not in df_merged.columns and 'Final_Estimate' in df_merged.columns:
    df_merged['Abs_Error'] = np.abs(df_merged['Error'])
if 'Pct_Error' not in df_merged.columns and 'Final_Estimate' in df_merged.columns:
    df_merged['Pct_Error'] = (df_merged['Abs_Error'] / df_merged['Actual_Count']) * 100.0

print(f'✅ Nạp thành công {len(df_merged)} mẫu thử nghiệm!')
print(f'   • Các loại đường kính ly có trong tập dữ liệu: {df_merged["Inner_Diameter_mm"].unique().tolist() if "Inner_Diameter_mm" in df_merged else "N/A"}')
df_merged[['Sample_ID', 'Actual_Count', 'Final_Estimate', 'CI90_Lower', 'CI90_Upper', 'Abs_Error', 'Pct_Error']].head(5)


In [ ]:
# =============================================================================
# 3. BẢNG CHỈ SỐ ĐO LƯỜNG ĐỘ CHÍNH XÁC & SO SÁNH ĐỐI ĐẦU 4 PHƯƠNG PHÁP
# =============================================================================
valid_data = df_merged.dropna(subset=['Final_Estimate']).copy()
y_true = valid_data['Actual_Count'].values

# Hàm tính toán bộ chỉ số khoa học cho một phương pháp ước lượng
def evaluate_estimator(pred_series, name):
    preds = pred_series.values
    valid_mask = ~np.isnan(preds) & ~np.isnan(y_true)
    yt, yp = y_true[valid_mask], preds[valid_mask]
    if len(yt) == 0: return None
    
    ss_res = np.sum((yt - yp) ** 2)
    ss_tot = np.sum((yt - np.mean(yt)) ** 2)
    r2 = 1.0 - (ss_res / ss_tot) if ss_tot != 0 else 0.0
    mae = np.mean(np.abs(yt - yp))
    rmse = np.sqrt(np.mean((yt - yp) ** 2))
    mape = np.mean(np.abs((yt - yp) / yt)) * 100.0
    tol_3s = np.mean(np.abs(yt - yp) <= 3) * 100.0
    tol_5s = np.mean(np.abs(yt - yp) <= 5) * 100.0
    tol_5p = np.mean(np.abs((yt - yp) / yt) <= 0.05) * 100.0
    tol_10p = np.mean(np.abs((yt - yp) / yt) <= 0.10) * 100.0
    
    return {
        'Phương pháp': name,
        'R² Score': round(r2, 4),
        'MAE (hạt)': round(mae, 2),
        'RMSE (hạt)': round(rmse, 2),
        'MAPE (%)': f'{mape:.2f}%',
        'Sai số <= 3 hạt': f'{tol_3s:.1f}%',
        'Sai số <= 5 hạt': f'{tol_5s:.1f}%',
        'Dung sai <= 5%': f'{tol_5p:.1f}%',
        'Dung sai <= 10%': f'{tol_10p:.1f}%'
    }

metrics_list = []
# 1. Bayesian Hybrid SOTA
m_bayes = evaluate_estimator(valid_data['Final_Estimate'], '✨ Bayesian Hybrid (FromThanh)')
if m_bayes: metrics_list.append(m_bayes)

# 2. Extra Trees Regression
if 'Regression_Estimate' in valid_data and valid_data['Regression_Estimate'].notna().any():
    m_reg = evaluate_estimator(valid_data['Regression_Estimate'], 'Mô hình Hồi quy Extra Trees (Cũ)')
    if m_reg: metrics_list.append(m_reg)

# 3. Geometry
if 'Geometry_Estimate' in valid_data and valid_data['Geometry_Estimate'].notna().any():
    m_geo = evaluate_estimator(valid_data['Geometry_Estimate'], 'Thuần Hình Học (Geometry 3D)')
    if m_geo: metrics_list.append(m_geo)

# 4. Weight
if 'Weight_Estimate' in valid_data and valid_data['Weight_Estimate'].notna().any():
    m_wt = evaluate_estimator(valid_data['Weight_Estimate'], 'Cân Mẫu (Weight-based)')
    if m_wt: metrics_list.append(m_wt)

df_metrics = pd.DataFrame(metrics_list)

# Đánh giá thêm độ phủ của dải tin cậy 90% (CI90)
if 'In_CI90' in valid_data and valid_data['In_CI90'].notna().any():
    ci90_cov = float(valid_data['In_CI90'].mean() * 100.0)
    ci90_w_avg = float((valid_data['CI90_Upper'] - valid_data['CI90_Lower']).mean())
    print('=' * 85)
    print(f'🌟 ĐỘ PHỦ KHOẢNG TIN CẬY 90% (CI90 COVERAGE PROBABILITY): {ci90_cov:.1f}%')
    print(f'   • Bề rộng dải tin cậy trung bình: ±{ci90_w_avg/2.0:.1f} hạt')
    print('=' * 85)

print('\n🏆 BẢNG TỔNG KẾT ĐỐI ĐẦU ĐA PHƯƠNG PHÁP:')
try:
    from IPython.display import display
    display(df_metrics.style.set_properties(**{'text-align': 'center'}).set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#1e293b'), ('color', '#38bdf8'), ('font-weight', 'bold')]}
    ]))
except (ImportError, NameError):
    print(df_metrics.to_string(index=False))


In [ ]:
# =============================================================================
# 4. CỤM BIỂU ĐỒ 1: TƯƠNG QUAN & ĐỘ CHÍNH XÁC (ACCURACY & CORRELATION)
# =============================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# --- 4.1. Đồ thị Tương quan Thực tế vs Dự đoán (Scatter Plot + Dải dung sai) ---
ax1 = axes[0]
y_act = valid_data['Actual_Count']
y_est = valid_data['Final_Estimate']

# Vẽ các điểm dữ liệu phân nhóm theo đường kính ly nếu có
if 'Lot_Group' in valid_data.columns:
    palette = {'Ly nhỏ (17.8mm)': '#8b5cf6', 'Ly vừa (22.5mm)': '#06b6d4', 'Ly lớn (32.6mm)': '#f59e0b'}
    sns.scatterplot(data=valid_data, x='Actual_Count', y='Final_Estimate', hue='Lot_Group',
                    palette=palette, s=70, alpha=0.9, ax=ax1, edgecolor='white')
else:
    sns.scatterplot(data=valid_data, x='Actual_Count', y='Final_Estimate', color='#7c3aed', s=70, alpha=0.9, ax=ax1)

min_val = min(y_act.min(), y_est.min()) - 10
max_val = max(y_act.max(), y_est.max()) + 10
ax1.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=1.5, label='Đường lý tưởng (y = x)')
ax1.fill_between([min_val, max_val], [min_val * 0.95, max_val * 0.95], [min_val * 1.05, max_val * 1.05],
                 color='red', alpha=0.08, label='Dải dung sai ±5%')
ax1.set_xlim(min_val, max_val)
ax1.set_ylim(min_val, max_val)
ax1.set_title(f'Tương Quan Thực Tế vs Bayes\n(R² = {m_bayes["R² Score"]}, MAE = {m_bayes["MAE (hạt)"]} hạt)', fontweight='bold', fontsize=12)
ax1.set_xlabel('Số Hạt Thực Tế (Ground Truth)')
ax1.set_ylabel('Số Hạt Dự Đoán (Bayesian Hybrid)')
ax1.legend(loc='upper left', frameon=True, framealpha=0.9)

# --- 4.2. Đồ thị Thỏa hiệp Bland-Altman (Mean vs Difference) ---
ax2 = axes[1]
mean_counts = (y_act + y_est) / 2.0
diff_counts = y_est - y_act
mean_diff = np.mean(diff_counts)
std_diff = np.std(diff_counts)
loa_upper = mean_diff + 1.96 * std_diff
loa_lower = mean_diff - 1.96 * std_diff

ax2.scatter(mean_counts, diff_counts, color='#0284c7', alpha=0.75, s=55, edgecolors='none')
ax2.axhline(mean_diff, color='#dc2626', linestyle='-', linewidth=1.5, label=f'Độ lệch trung bình: {mean_diff:+.2f} hạt')
ax2.axhline(loa_upper, color='#64748b', linestyle='--', linewidth=1.2, label=f'+1.96 SD: {loa_upper:+.2f}')
ax2.axhline(loa_lower, color='#64748b', linestyle='--', linewidth=1.2, label=f'-1.96 SD: {loa_lower:+.2f}')
ax2.fill_between([mean_counts.min() - 10, mean_counts.max() + 10], loa_lower, loa_upper, color='#0284c7', alpha=0.05)
ax2.set_xlim(mean_counts.min() - 10, mean_counts.max() + 10)
ax2.set_title('Đồ Thị Thỏa Hiệp Bland-Altman\n(Mức độ đồng thuận sai số)', fontweight='bold', fontsize=12)
ax2.set_xlabel('Số Hạt Trung Bình [(Thực tế + Dự đoán) / 2]')
ax2.set_ylabel('Sai Số Phần Dư (Dự đoán - Thực tế)')
ax2.legend(loc='upper right', frameon=True, fontsize=9)

# --- 4.3. Cumulative Error Distribution Curve (Đường cong tích lũy sai số) ---
ax3 = axes[2]
thresholds = np.linspace(0, 15, 60)
methods_curves = [('Bayesian Hybrid', valid_data['Final_Estimate'], '#7c3aed', 2.2)]
if 'Regression_Estimate' in valid_data and valid_data['Regression_Estimate'].notna().any():
    methods_curves.append(('Extra Trees', valid_data['Regression_Estimate'], '#16a34a', 1.6))
if 'Geometry_Estimate' in valid_data and valid_data['Geometry_Estimate'].notna().any():
    methods_curves.append(('Thuần Hình Học', valid_data['Geometry_Estimate'], '#ea580c', 1.6))
if 'Weight_Estimate' in valid_data and valid_data['Weight_Estimate'].notna().any():
    methods_curves.append(('Cân Mẫu', valid_data['Weight_Estimate'], '#2563eb', 1.6))

for name, srs, col, lw in methods_curves:
    abs_errs = np.abs(srs.values - y_true)
    cum_pct = [np.mean(abs_errs <= t) * 100.0 for t in thresholds]
    ax3.plot(thresholds, cum_pct, label=name, color=col, linewidth=lw)

ax3.axvline(3, color='#94a3b8', linestyle=':', label='Ngưỡng sai số 3 hạt')
ax3.axvline(5, color='#cbd5e1', linestyle=':', label='Ngưỡng sai số 5 hạt')
ax3.set_title('Đường Cong Tích Lũy Sai Số\n(Tỷ lệ mẫu đạt dung sai theo số hạt)', fontweight='bold', fontsize=12)
ax3.set_xlabel('Ngưỡng Sai Số Tuyệt Đối (Số hạt)')
ax3.set_ylabel('Tỷ Lệ Mẫu Đạt (%)')
ax3.set_ylim(0, 102)
ax3.legend(loc='lower right', frameon=True, fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_PLOTS_DIR / '1_accuracy_correlation_analytics.png', dpi=300)
plt.show()


In [ ]:
# =============================================================================
# 5. CỤM BIỂU ĐỒ 2: PHÂN PHỐI SAI SỐ & ĐỘ BẤT ĐỊNH (UNCERTAINTY ANALYTICS)
# =============================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# --- 5.1. Phân phối sai số phần dư (Residuals Density) ---
ax1 = axes[0]
res_bayes = valid_data['Final_Estimate'] - y_true
sns.histplot(res_bayes, kde=True, color='#7c3aed', bins=14, ax=ax1, stat='density', alpha=0.45, label='Bayesian Hybrid')
if 'Regression_Estimate' in valid_data and valid_data['Regression_Estimate'].notna().any():
    res_reg = valid_data['Regression_Estimate'] - y_true
    sns.kdeplot(res_reg, color='#16a34a', linewidth=1.8, ax=ax1, label='Extra Trees')
if 'Geometry_Estimate' in valid_data and valid_data['Geometry_Estimate'].notna().any():
    res_geo = valid_data['Geometry_Estimate'] - y_true
    sns.kdeplot(res_geo, color='#ea580c', linewidth=1.8, ax=ax1, label='Geometry 3D')

ax1.axvline(0, color='black', linestyle='--', linewidth=1.2)
ax1.set_title('Phân Phối Sai Số Phần Dư\n(So sánh độ tập trung quanh 0)', fontweight='bold', fontsize=12)
ax1.set_xlabel('Sai Số (Dự Đoán - Thực Tế) [hạt]')
ax1.set_ylabel('Mật Độ Xác Suất (Density)')
ax1.legend(loc='upper right', frameon=True, fontsize=9)

# --- 5.2. Kiểm tra Độ phủ & Bề rộng Khoảng tin cậy 90% (CI90 Coverage) ---
ax2 = axes[1]
if 'CI90_Lower' in valid_data and 'CI90_Upper' in valid_data:
    # Sắp xếp mẫu theo số hạt thực tế tăng dần
    sorted_df = valid_data.sort_values(by='Actual_Count').reset_index(drop=True)
    sample_idx = np.arange(len(sorted_df))
    
    # Vẽ dải tin cậy 90%
    ax2.fill_between(sample_idx, sorted_df['CI90_Lower'], sorted_df['CI90_Upper'], color='#c4b5fd', alpha=0.45, label='Dải Tin Cậy 90% (CI90)')
    ax2.plot(sample_idx, sorted_df['Final_Estimate'], color='#7c3aed', linewidth=1.5, label='Bayes Dự Đoán')
    
    # Điểm thực tế: Xanh nếu nằm trong CI90, Đỏ nếu lọt ra ngoài
    in_mask = sorted_df['In_CI90'] == True
    ax2.scatter(sample_idx[in_mask], sorted_df.loc[in_mask, 'Actual_Count'], color='#10b981', s=35, label='Thực tế (Nằm trong CI90)', zorder=4)
    if (~in_mask).any():
        ax2.scatter(sample_idx[~in_mask], sorted_df.loc[~in_mask, 'Actual_Count'], color='#ef4444', s=55, marker='x', label='Thực tế (Ngoài CI90)', zorder=5)
    
    ax2.set_title(f'Kiểm Tra Độ Phủ Khoảng Tin Cậy 90%\n(Độ phủ đạt: {ci90_cov:.1f}% / 90% mục tiêu)', fontweight='bold', fontsize=12)
    ax2.set_xlabel('Mẫu Thử Nghiệm (Sắp xếp theo số hạt tăng dần)')
    ax2.set_ylabel('Số Hạt')
    ax2.legend(loc='upper left', frameon=True, fontsize=9)
else:
    ax2.text(0.5, 0.5, 'Thiếu thông tin CI90', ha='center', va='center')

# --- 5.3. Phân tích Đa nhánh: Ảnh thuần (Vision) vs Có cân (Weight) ---
ax3 = axes[2]
if 'Vision_Only_Est' in valid_data and 'Weight_Assisted_Est' in valid_data and valid_data['Vision_Only_Est'].notna().any():
    sns.scatterplot(data=valid_data, x='Vision_Only_Est', y='Weight_Assisted_Est', hue='Lot_Group' if 'Lot_Group' in valid_data else None,
                    palette='viridis', s=65, alpha=0.85, ax=ax3)
    b_min = min(valid_data['Vision_Only_Est'].min(), valid_data['Weight_Assisted_Est'].min()) - 10
    b_max = max(valid_data['Vision_Only_Est'].max(), valid_data['Weight_Assisted_Est'].max()) + 10
    ax3.plot([b_min, b_max], [b_min, b_max], 'k--', linewidth=1.2, label='Đường đồng thuận (Vision = Weight)')
    ax3.set_xlim(b_min, b_max)
    ax3.set_ylim(b_min, b_max)
    ax3.set_title('Phân Tích Đa Nhánh Bayes\n(Ảnh Thị Giác vs Kết Hợp Cân)', fontweight='bold', fontsize=12)
    ax3.set_xlabel('Dự Đoán Nhánh Ảnh Thuần (Vision-Only)')
    ax3.set_ylabel('Dự Đoán Nhánh Có Cân (Weight-Assisted)')
    ax3.legend(loc='upper left', frameon=True, fontsize=9)
else:
    ax3.text(0.5, 0.5, 'Không có dữ liệu 2 nhánh', ha='center', va='center')

plt.tight_layout()
plt.savefig(OUTPUT_PLOTS_DIR / '2_distribution_uncertainty_analytics.png', dpi=300)
plt.show()


In [ ]:
# =============================================================================
# 6. CỤM BIỂU ĐỒ 3: PHÂN TÍCH SAI SỐ THEO KÍCH CỠ CỐC & HÌNH HỌC VẬT LÝ
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

# --- 6.1. So sánh MAE theo từng kích cỡ ly (Lot Group Breakdown) ---
ax1 = axes[0]
if 'Lot_Group' in valid_data.columns and 'Regression_Estimate' in valid_data:
    lot_metrics = []
    for group, g_df in valid_data.groupby('Lot_Group'):
        mae_b = np.mean(np.abs(g_df['Final_Estimate'] - g_df['Actual_Count']))
        mae_r = np.mean(np.abs(g_df['Regression_Estimate'] - g_df['Actual_Count']))
        lot_metrics.append({'Nhóm ly': group, 'Bayesian Hybrid (FromThanh)': mae_b, 'Extra Trees (Cũ)': mae_r, 'Số mẫu': len(g_df)})
    
    df_lot_mae = pd.DataFrame(lot_metrics)
    df_melt = pd.melt(df_lot_mae, id_vars=['Nhóm ly', 'Số mẫu'], value_vars=['Bayesian Hybrid (FromThanh)', 'Extra Trees (Cũ)'],
                      var_name='Mô hình', value_name='MAE (hạt)')
    
    barplot = sns.barplot(data=df_melt, x='Nhóm ly', y='MAE (hạt)', hue='Mô hình', palette=['#7c3aed', '#16a34a'], ax=ax1, edgecolor='black', linewidth=0.5)
    for p in barplot.patches:
        h = p.get_height()
        if h > 0: ax1.annotate(f'{h:.2f}', (p.get_x() + p.get_width() / 2., h), ha='center', va='bottom', fontsize=9, fontweight='bold', xytext=(0, 2), textcoords='offset points')
    
    ax1.set_title('Sai Số MAE Theo Từng Loại Ly\n(Minh chứng khả năng thích nghi đa kích cỡ của Bayes)', fontweight='bold', fontsize=12)
    ax1.set_ylabel('MAE (Số hạt sai lệch)')
    ax1.legend(loc='upper right', frameon=True)
else:
    ax1.text(0.5, 0.5, 'Thiếu thông tin phân nhóm ly', ha='center', va='center')

# --- 6.2. Sai số phần trăm (MAPE %) theo Chiều cao lớp lúa (Rice Height) ---
ax2 = axes[1]
if 'Rice_Height_mm' in valid_data.columns:
    sns.scatterplot(data=valid_data, x='Rice_Height_mm', y='Pct_Error', hue='Lot_Group' if 'Lot_Group' in valid_data else None,
                    palette='tab10', s=70, alpha=0.85, ax=ax2)
    ax2.axhline(5.0, color='red', linestyle='--', linewidth=1.2, label='Ngưỡng chuẩn 5%')
    ax2.axhline(10.0, color='orange', linestyle=':', linewidth=1.2, label='Ngưỡng dung sai 10%')
    ax2.set_title('Sai Số Tương Đối (%) Theo Mức Lúa Trong Ly\n(Đánh giá độ ổn định từ ly vơi đến ly đầy)', fontweight='bold', fontsize=12)
    ax2.set_xlabel('Chiều Cao Lớp Lúa (mm)')
    ax2.set_ylabel('Sai Số Tương Đối (% Error)')
    ax2.legend(loc='upper right', frameon=True, fontsize=9)
else:
    ax2.text(0.5, 0.5, 'Thiếu thông tin chiều cao lúa', ha='center', va='center')

plt.tight_layout()
plt.savefig(OUTPUT_PLOTS_DIR / '3_lot_size_geometry_breakdown.png', dpi=300)
plt.show()


In [ ]:
# =============================================================================
# 7. THƯ VIỆN HÌNH ẢNH MẪU THỰC ĐỊA: TOP MẪU CHUẨN XÁC & OUTLIERS
# =============================================================================
top_best = valid_data.sort_values(by='Abs_Error').head(4)
top_worst = valid_data.sort_values(by='Abs_Error', ascending=False).head(4)

def render_sample_cards(samples_df, title, is_best=True):
    fig, axes = plt.subplots(1, 4, figsize=(18, 4.8))
    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.02, color='#047857' if is_best else '#b91c1c')
    
    for idx, (_, row) in enumerate(samples_df.iterrows()):
        ax = axes[idx]
        sid = str(row['Sample_ID']).strip()
        img_path = RAW_IMG_DIR / f'{sid}.jpg'
        
        if img_path.is_file():
            with Image.open(img_path) as im:
                ax.imshow(im)
        else:
            ax.text(0.5, 0.5, f'Không tìm thấy ảnh\n{sid}.jpg', ha='center', va='center')
            ax.set_facecolor('#f1f5f9')
        
        act = row['Actual_Count']
        est = row['Final_Estimate']
        err = row['Abs_Error']
        pct = row['Pct_Error']
        ci_lo = row.get('CI90_Lower', 'N/A')
        ci_hi = row.get('CI90_Upper', 'N/A')
        diam = row.get('Inner_Diameter_mm', 'N/A')
        
        # Overlay thông số
        card_text = (
            f"Mẫu: {sid} (D={diam}mm)\n"
            f"Thực tế: {act:.0f} hạt\n"
            f"Bayes: {est:.1f} hạt\n"
            f"CI90: [{ci_lo} ~ {ci_hi}]\n"
            f"Lệch: {err:.1f} hạt ({pct:.1f}%)"
        )
        box_color = '#ecfdf5' if is_best else '#fef2f2'
        border_color = '#10b981' if is_best else '#ef4444'
        ax.text(0.04, 0.04, card_text, transform=ax.transAxes,
                fontsize=9, fontweight='bold', va='bottom', ha='left',
                bbox=dict(boxstyle='round,pad=0.4', facecolor=box_color, edgecolor=border_color, alpha=0.92))
        
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_color(border_color)
            spine.set_linewidth(2)
    
    plt.tight_layout()
    file_name = '4_top_best_samples.png' if is_best else '5_top_worst_samples.png'
    plt.savefig(OUTPUT_PLOTS_DIR / file_name, dpi=300, bbox_inches='tight')
    plt.show()

print('🌟 HIỂN THỊ HÌNH ẢNH MẪU THỰC ĐỊA:')
render_sample_cards(top_best, '🌟 TOP 4 MẪU ĐẠT ĐỘ CHÍNH XÁC CAO NHẤT (SAI SỐ CỰC THẤP)', is_best=True)
render_sample_cards(top_worst, '⚠️ TOP 4 MẪU CẦN LƯU Ý (OUTLIERS / SAI SỐ CAO NHẤT ĐỂ PHÂN TÍCH)', is_best=False)


In [ ]:
# =============================================================================
# 8. CÔNG CỤ SOI CHI TIẾT MỘT MẪU BẤT KỲ (SINGLE SAMPLE INSPECTOR)
# =============================================================================
# Bạn có thể thay đổi mã mẫu tại đây để kiểm tra chi tiết (hoặc để None để lấy ngẫu nhiên):
SAMPLE_ID_TO_INSPECT = None  # Ví dụ: 'M0015' hoặc None

if SAMPLE_ID_TO_INSPECT is None:
    inspect_row = valid_data.sample(1).iloc[0]
else:
    matches = valid_data[valid_data['Sample_ID'] == str(SAMPLE_ID_TO_INSPECT).strip()]
    inspect_row = matches.iloc[0] if len(matches) > 0 else valid_data.iloc[0]

sid = str(inspect_row['Sample_ID']).strip()
img_path = RAW_IMG_DIR / f'{sid}.jpg'

fig = plt.figure(figsize=(15, 5))
gs = fig.add_gridspec(1, 2, width_ratios=[1.1, 1.4])

# Cột 1: Hiển thị hình ảnh gốc
ax_img = fig.add_subplot(gs[0])
if img_path.is_file():
    with Image.open(img_path) as im:
        ax_img.imshow(im)
    ax_img.set_title(f'Ảnh Chụp Thực Tế: {sid}', fontweight='bold', fontsize=12)
else:
    ax_img.text(0.5, 0.5, f'Không có ảnh {sid}.jpg', ha='center', va='center')
ax_img.set_xticks([])
ax_img.set_yticks([])

# Cột 2: So sánh đối chứng các phương pháp cho mẫu này
ax_bar = fig.add_subplot(gs[1])
methods_compare = [
    ('Thực tế (Ground Truth)', inspect_row['Actual_Count'], '#10b981'),
    ('Bayes Hybrid', inspect_row['Final_Estimate'], '#7c3aed')
]
if 'Regression_Estimate' in inspect_row and pd.notna(inspect_row['Regression_Estimate']):
    methods_compare.append(('Extra Trees', inspect_row['Regression_Estimate'], '#16a34a'))
if 'Geometry_Estimate' in inspect_row and pd.notna(inspect_row['Geometry_Estimate']):
    methods_compare.append(('Geometry 3D', inspect_row['Geometry_Estimate'], '#ea580c'))
if 'Weight_Estimate' in inspect_row and pd.notna(inspect_row['Weight_Estimate']):
    methods_compare.append(('Cân Mẫu', inspect_row['Weight_Estimate'], '#2563eb'))

m_names = [m[0] for m in methods_compare]
m_vals = [float(m[1]) for m in methods_compare]
m_colors = [m[2] for m in methods_compare]

bars = ax_bar.barh(m_names, m_vals, color=m_colors, height=0.55)
for bar in bars:
    w = bar.get_width()
    ax_bar.text(w + 2, bar.get_y() + bar.get_height()/2.0, f'{w:.1f} hạt', va='center', fontsize=10, fontweight='bold')

# Đường gióng thẳng đứng đối chiếu số hạt thực tế
ax_bar.axvline(inspect_row['Actual_Count'], color='#10b981', linestyle='--', linewidth=1.5, label='Mức thực tế chuẩn')
ax_bar.set_xlim(0, max(m_vals) * 1.18)
ci_lo = inspect_row.get('CI90_Lower', 'N/A')
ci_hi = inspect_row.get('CI90_Upper', 'N/A')
ax_bar.set_title(f'So Sánh Các Mô Hình Cho Mẫu {sid}\n[CI90 Bayes: {ci_lo} ~ {ci_hi} hạt | Lệch: {inspect_row["Abs_Error"]:.1f} hạt]', fontweight='bold', fontsize=12)
ax_bar.set_xlabel('Số Hạt Dự Đoán')
ax_bar.legend(loc='lower right')

plt.tight_layout()
plt.show()


In [ ]:
# =============================================================================
# 9. XUẤT TOÀN BỘ BỘ SƯU TẬP ĐỒ THỊ & ĐÓNG GÓI BÁO CÁO (EXPORT GALLERY)
# =============================================================================
import shutil

zip_gallery_path = BENCHMARK_DIR / 'visual_analytics_bundle'
shutil.make_archive(str(zip_gallery_path), 'zip', str(OUTPUT_PLOTS_DIR))

print('=' * 80)
print('🎉 ĐÃ HOÀN TẤT TRỰC QUAN HÓA TOÀN DIỆN KẾT QUẢ BENCHMARK!')
print('=' * 80)
print(f'   📂 Thư mục chứa đồ thị độ phân giải cao: {OUTPUT_PLOTS_DIR}')
for img_f in OUTPUT_PLOTS_DIR.glob('*.png'):
    print(f'      • 📸 {img_f.name}')
print(f'   📦 Gói ZIP toàn bộ đồ thị: {zip_gallery_path}.zip')
print('=' * 80)

# Hướng dẫn tải về trên Google Colab nếu đang chạy Colab
if IS_COLAB:
    print('\n💡 [Google Colab] Bạn có thể tải gói ZIP đồ thị về máy tính bằng mã sau:')
    print('from google.colab import files')
    print(f"files.download('{zip_gallery_path}.zip')")
